In [30]:
import io
import os
import fnmatch
from typing import Optional
import paramiko
import pandas as pd

def fetch_and_concat_csvs(
    ip: str,
    user: str,
    password: str,
    remote_dir: str,
    pattern: str = "df_metrics*",
    port: int = 22,
    return_csv_string: bool = False,
    timeout: Optional[float] = 10.0,
) -> pd.DataFrame:
    """
    Se connecte en SFTP et télécharge/télécharge en mémoire tous les fichiers
    du dossier `remote_dir` qui correspondent au `pattern` (fnmatch), lit les CSV,
    ajoute une colonne 'file' contenant le nom du fichier, concatène et renvoie
    le DataFrame final.

    Paramètres
    ----------
    ip, user, password : str
        Informations de connexion SSH.
    remote_dir : str
        Dossier distant à parcourir (chemin absolu ou relatif à l'utilisateur SFTP).
    pattern : str
        Pattern shell-style (ex: 'df_metrics*').
    port : int
        Port SSH (par défaut 22).
    return_csv_string : bool
        Si True, renvoie une tuple (DataFrame, csv_string). Sinon seulement DataFrame.
    timeout : float|None
        Timeout connexion en secondes.

    Retour
    ------
    pd.DataFrame
        DataFrame pandas concaténé contenant tous les enregistrements et une colonne 'file'.
        (ou (DataFrame, csv_string) si return_csv_string=True)
    """

    # Prépare le client SSH / SFTP
    transport = None
    sftp = None
    try:
        # Connexion SSH (transport)
        transport = paramiko.Transport((ip, port))
        transport.connect(username=user, password=password)
        # Ouvrir SFTP
        sftp = paramiko.SFTPClient.from_transport(transport)

        # Lister fichiers dans le répertoire distant
        try:
            files = sftp.listdir(remote_dir)
        except IOError:
            # Peut-être remote_dir non trouvé ; relancer en absolu ou lever l'erreur
            raise

        # Filtrer par pattern
        matched = [f for f in files if fnmatch.fnmatch(f, pattern)]
        if not matched:
            # Aucun fichier trouvé : renvoyer DataFrame vide
            empty_df = pd.DataFrame()
            if return_csv_string:
                return empty_df, ""
            return empty_df

        dfs = []
        for fname in matched:
            if 'volting' in fname or 'federated' in fname:
                continue
            #if 'tree' not in fname:
            #    continue
            remote_path = os.path.join(remote_dir, fname)
            # Ouvre en binaire, lit tout en mémoire
            with sftp.open(remote_path, "rb") as remote_file:
                raw = remote_file.read()
            # Tenter de lire avec pandas
            # On essaie d'abord BytesIO (pandas gère souvent les bytes),
            # sinon on décode en utf-8 en tolérant les erreurs.
            df = None
            try:
                df = pd.read_csv(io.BytesIO(raw))
            except Exception:
                # try decoding then StringIO
                try:
                    txt = raw.decode("utf-8", errors="replace")
                    df = pd.read_csv(io.StringIO(txt))
                except Exception as e:
                    # En cas d'échec, lever une erreur explicite
                    raise RuntimeError(f"Impossible de parser le CSV distant {remote_path!r}: {e}")

            # Ajouter colonne 'file' avec le nom du fichier (basename)
            df["file"] = os.path.basename(fname)
            dfs.append(df)

        # Concaténer
        final_df = pd.concat(dfs, ignore_index=True, sort=False)

        if return_csv_string:
            csv_str = final_df.to_csv(index=False)
            return final_df, csv_str

        return final_df

    finally:
        # Fermeture propre des connexions
        if sftp is not None:
            try:
                sftp.close()
            except Exception:
                pass
        if transport is not None:
            try:
                transport.close()
            except Exception:
                pass


IP = "mesohelios1.univ-fcomte.fr"
USER = "ncaron"
PASSWORD = "xp2nrfeu"
REMOTE_DIR = "/Home/Users/ncaron/WORK/GNN/bdiff/firepoint/2x2/test/occurence_default/full_all_departement_0_None_node"

df = fetch_and_concat_csvs(IP, USER, PASSWORD, REMOTE_DIR, pattern="df_metrics*")
print("Lignes totales :", len(df))
print(df.head())

Lignes totales : 162
   Unnamed: 0                                                Run  nbsinister  \
0           0  all_GRU_search_full_10_all_one_nbsinister-kmea...      2192.0   
1           0  all_GRU_search_full_10_all_one_nbsinister-kmea...      2192.0   
2           1  all_LSTM_search_full_10_all_one_nbsinister-kme...      2189.0   
3           2  all_NetMLP_search_full_0_all_one_nbsinister-km...      2189.0   
4           3  all_DilatedCNN_search_full_10_all_one_nbsinist...      2188.0   

         r2       mse  unknow_sample_proportion  iou_class_hard  \
0 -1.132346  0.197802                  0.137303           0.181   
1 -1.181599  0.202371                  0.137303           0.195   
2 -0.733231  0.161067                  0.137651           0.222   
3 -0.782531  0.165649                  0.137651           0.214   
4 -1.010819  0.187305                  0.137881           0.216   

   iou_wildfire_or_pred_class_hard  iou_wildfire_and_pred_class_hard  \
0                      

In [31]:
def select_metrics(df : pd.DataFrame, metrics : list[str]):
    return df[['Run'] + metrics]

#df = select_metrics(df, ['mean_iou_test', 'mean_f1_test', 'mean_prec_test', 'mean_recall_test', 'mean_normalized_iou_test', \
#                         'mean_normalized_f1_test', 'mean_normalized_prec_test', 'mean_normalized_rec_test', \
#                            'mean_iou_elt_sup_3.0_test', 'mean_f1_elt_sup_3.0_test', 'mean_prec_elt_sup_3.0_test', 'mean_rec_elt_sup_3.0_test'])

In [32]:
def parse_run_name(x):
    print(x)
    dico = {}
    vec = x.split('_')
    dico['Department'] = vec[0]
    i = 1
    dico['Model'] = vec[i]
    i += 1
    if dico['Model'] == 'fwi-mean-[5, 10.5, 21.5, 34.5]-5':
        dico['Number_of_features'] = vec[i]
        i += 1
        dico['weight'] = vec[i]
        i += 1
        dico['Target'] = vec[i]
        i += 1
        dico['Task_type'] = vec[i]
        i += 1
        dico['under_sampling'] = vec[i]
        dico['over_sampling'] = vec[i]
        i += 1
        dico['Number_of_features'] = vec[i]
    else:
        dico['under_sampling'] = vec[i]
        i += 1
        dico['over_sampling'] = vec[i]
        i += 1
        dico['kdays'] = vec[i]
        i += 1
        dico['Number_of_features'] = vec[i]
        i += 1
        dico['weight'] = vec[i]
        i += 1
        dico['Target'] = vec[i]
        i += 1

    if dico['Model'] != 'fwi-mean-[5, 10.5, 21.5, 34.5]-5':
        dico['Task_type'] = vec[i]
        i += 1
        dico['loss'] = vec[i]
        i += 1
        try:
            if vec[i] == 'departement':
                dico['scale_result'] = vec[i]
            else:
                int(vec[i])
                dico['scale_result'] = vec[i]
            i += 1
        except Exception as e:
            try:
                float(vec[i])
                return None
            except:
                pass
            pass
    else:
        dico['loss'] = None
    
    i += 1
    #dico['Number_of_features'] = vec[i]
    i += 1
    dico['Scale'] = vec[i]
    if 'scale_result' not in dico.keys():
        dico['scale_result'] = vec[i]
    i += 1
    dico['Days_in_futur'] = vec[i]
    i += 1
    dico['Base'] = vec[i]
    i += 1
    dico['Method'] = vec[i]
    i += 1

    if i == len(vec):
        return dico
    if vec[i] == 'kmeans':
        i += 1
        dico['kmeans_shift'] = vec[i]
        i += 1
        dico['kmeans_thresh'] = vec[i]
        i += 1
    return dico

# Initialisation des colonnes avec des valeurs None
df['Department'] = None
df['Model'] = None
df['Target'] = None
df['Task_type'] = None
df['Drop'] = None
df['Loss_function'] = None
df['under_sampling'] = None
df['over_sampling'] = None
df['kdays'] = None
df['Number_of_features'] = None
df['Scale'] = None
df['Base'] = None
df['Method'] = None
df['Days_in_futur'] = None
df['weight'] = None
df['kmeans_thresh'] = None
df['kmeans_shift'] = None
df['scale_result'] = None

# Boucle pour remplir les colonnes avec les valeurs de dico_parse
for index, row in df.iterrows():
    dico_parse = parse_run_name(row['Run'])
    if dico_parse is None:
        continue
    # Mise à jour de chaque colonne avec les valeurs du dictionnaire dico_parse
    df.loc[index, 'Department'] = dico_parse.get('Department')
    df.loc[index, 'scale_result'] = dico_parse.get('scale_result')
    df.loc[index, 'Drop'] = dico_parse.get('Drop')
    df.loc[index, 'Model'] = dico_parse.get('Model')
    df.loc[index, 'Target'] = dico_parse.get('Target')
    df.loc[index, 'Task_type'] = dico_parse.get('Task_type')
    df.loc[index, 'Loss_function'] = dico_parse.get('loss')
    df.loc[index, 'under_sampling'] = dico_parse.get('under_sampling')
    df.loc[index, 'over_sampling'] = dico_parse.get('over_sampling')
    df.loc[index, 'kdays'] = dico_parse.get('kdays')
    df.loc[index, 'Number_of_features'] = dico_parse.get('Number_of_features')
    df.loc[index, 'Scale'] = dico_parse.get('Scale')
    df.loc[index, 'Base'] = dico_parse.get('Base')
    df.loc[index, 'Method'] = dico_parse.get('Method')
    df.loc[index, 'Days_in_futur'] = dico_parse.get('Days_in_futur')

    df.loc[index, 'weight'] = dico_parse.get('weight')
    df.loc[index, 'kmeans_thresh'] = dico_parse.get('kmeans_thresh', 0)
    df.loc[index, 'kmeans_shift'] = dico_parse.get('kmeans_shift', 0)

/tmp/ipykernel_26516/2411463731.py:83: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Department'] = None
/tmp/ipykernel_26516/2411463731.py:84: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Model'] = None
/tmp/ipykernel_26516/2411463731.py:85: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[

all_GRU_search_full_10_all_one_nbsinister-kmeans-5-Class-Dept_regression_egpdCDFCluster-id{node}-NC{95}-G{False}_10_full_all_departement_0_None_node
all_GRU_search_full_10_all_one_nbsinister-kmeans-5-Class-Dept_classification_odwk_10_full_all_departement_0_None_node
all_LSTM_search_full_10_all_one_nbsinister-kmeans-5-Class-Dept_classification_odwk_10_full_all_departement_0_None_node
all_NetMLP_search_full_0_all_one_nbsinister-kmeans-5-Class-Dept_classification_odwk_10_full_all_departement_0_None_node
all_DilatedCNN_search_full_10_all_one_nbsinister-kmeans-5-Class-Dept_classification_odwk_10_full_all_departement_0_None_node
all_GRU_search_full_10_all_one_nbsinister-kmeans-5-Class-Dept_classification_fdwk_10_full_all_departement_0_None_node
all_LSTM_search_full_10_all_one_nbsinister-kmeans-5-Class-Dept_classification_fdwk_10_full_all_departement_0_None_node
all_NetMLP_search_full_0_all_one_nbsinister-kmeans-5-Class-Dept_classification_fdwk_10_full_all_departement_0_None_node
all_DilatedC

In [33]:
df

,Unnamed: 0,Run,nbsinister,r2,mse,unknow_sample_proportion,iou_class_hard,iou_wildfire_or_pred_class_hard,iou_wildfire_and_pred_class_hard,rec_bin_class_hard,...,kdays,Number_of_features,Scale,Base,Method,Days_in_futur,weight,kmeans_thresh,kmeans_shift,scale_result
0,0,all_GRU_search_full_10_all_one_nbsinister-kmea...,2192.0,-1.132346,0.197802,0.137303,0.181,0.181,0.731,0.738,...,10,all,departement,None,node,0,one,0,0,10
1,0,all_GRU_search_full_10_all_one_nbsinister-kmea...,2192.0,-1.181599,0.202371,0.137303,0.195,0.195,0.621,0.452,...,10,all,departement,None,node,0,one,0,0,10
2,1,all_LSTM_search_full_10_all_one_nbsinister-kme...,2189.0,-0.733231,0.161067,0.137651,0.222,0.222,0.723,0.718,...,10,all,departement,None,node,0,one,0,0,10
3,2,all_NetMLP_search_full_0_all_one_nbsinister-km...,2189.0,-0.782531,0.165649,0.137651,0.214,0.214,0.728,0.719,...,0,all,departement,None,node,0,one,0,0,10
4,3,all_DilatedCNN_search_full_10_all_one_nbsinist...,2188.0,-1.010819,0.187305,0.137881,0.216,0.216,0.725,0.753,...,10,all,departement,None,node,0,one,0,0,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157,1,all_LSTM_search_full_10_all_one_nbsinister-kme...,2189.0,-0.415842,0.131573,0.137651,0.243,0.243,0.726,0.637,...,10,all,departement,None,node,0,one,0,0,10
158,2,all_NetMLP_search_full_0_all_one_nbsinister-km...,2189.0,-1.395074,0.222572,0.137651,0.201,0.201,0.652,0.691,...,0,all,departement,None,node,0,one,0,0,10
159,3,all_DilatedCNN_search_full_10_all_one_nbsinist...,2188.0,-0.624322,0.151303,0.137881,0.234,0.234,0.714,0.656,...,10,all,departement,None,node,0,one,0,0,10
160,0,all_GRU_search_full_10_all_one_nbsinister-kmea...,2188.0,NaN,NaN,0.133271,0.232,0.232,0.726,NaN,...,10,all,departement,None,node,0,one,0,0,10


In [34]:
def select_models(df : pd.DataFrame, models : list[str]):
    return df[df['Model'].isin(models)]

df = select_models(df, ['GRU', 'LSTM', 'NetMLP', 'DilatedCNN', 'Dualtraining', 'lg', 'xgboost', 'catboost', 'graphCastGRU'])
df = df.dropna(subset='mean_iou_test')

In [35]:
df.Loss_function.unique()

array(['egpdCDFCluster-id{node}-NC{95}-G{False}', 'odwk', 'fdwk',
       'wkloss', 'ordidice', 'l2', 'softmax', 'mcewk-C{0.5}',
       'weightedcrossentropy', 'egpdLogCDF', 'fdice', 'bceloss', 'gwdl'],
      dtype=object)

In [ ]:
import pandas as pd
import numpy as np

def to_latex(df: pd.DataFrame, metrics: list[list[str]], loss: str, targets: list[str]) -> str:
    """
    Génère un tableau LaTeX :
    - une ligne par modèle
    - chaque métrique est affichée comme "mean ± std" avec 2 chiffres après la virgule
    - NaN si la métrique pour un modèle/target n'existe pas
    - filtrage par Loss_function
    """

    # Filtrer par loss
    df = df[df["Loss_function"] == loss]

    if df.empty:
        raise ValueError(f"Aucune ligne trouvée avec Loss_function == '{loss}'")

    # Liste des modèles uniques
    models = df["Model"].unique()

    # Colonnes de métriques
    ordered_metrics = [m for group in metrics for m in group]

    col_format = "l" + "c" * len(ordered_metrics)

    # En-tête
    target_header = ["Model"]
    for target, group in zip(targets, metrics):
        target_header.extend([target] * len(group))
    target_header = " & ".join(target_header) + " \\\\"

    metric_header = [""] + ordered_metrics
    metric_header = " & ".join(metric_header) + " \\\\"

    rows = []

    for model in models:
        values = [model]

        for target_group, target in zip(metrics, targets):
            for m in target_group:
                # colonne std correspondante
                std_col = "std_" + (m[5:] if m.startswith("mean_") else m)

                # Filtrer sur modèle et target exact
                matching_rows = df[(df["Model"] == model) & (df["Target"] == target)]

                if not matching_rows.empty:
                    mean_val = matching_rows.iloc[0].get(m, np.nan)
                    std_val = matching_rows.iloc[0].get(std_col, np.nan)
                    if pd.isna(mean_val) or pd.isna(std_val):
                        val = "NaN"
                    else:
                        val = f"{mean_val:.2f} ± {std_val:.2f}"  # deux chiffres après la virgule
                else:
                    val = "NaN"

                values.append(val)
                 
        rows.append(" & ".join(values) + " \\\\")

    # Construction du tableau LaTeX
    latex = []
    latex.append("\\begin{tabular}{" + col_format + "}")
    latex.append("\\hline")
    latex.append(target_header)
    latex.append(metric_header)
    latex.append("\\hline")
    latex.extend(rows)
    latex.append("\\hline")
    latex.append("\\end{tabular}")
    
    return "\n".join(latex)

print(to_latex(df, metrics=[['mean_f1_test', 'mean_prec_test', 'mean_recall_test', 'mean_iou_test', 'mean_auoc_test'], \
                            ['mean_iou_test', 'mean_auoc_test']], loss='softmax',\
               targets=['nbsinister-kmeans-5-Class-Dept', 'burnedarea-kmeans-5-Class-Dept']))

\begin{tabular}{lccccccc}
\hline
Model & nbsinister-kmeans-5-Class-Dept & nbsinister-kmeans-5-Class-Dept & nbsinister-kmeans-5-Class-Dept & nbsinister-kmeans-5-Class-Dept & nbsinister-kmeans-5-Class-Dept & burnedarea-kmeans-5-Class-Dept & burnedarea-kmeans-5-Class-Dept \\
 & mean_f1_test & mean_prec_test & mean_recall_test & mean_iou_test & mean_auoc_test & mean_iou_test & mean_auoc_test \\
\hline
catboost & 0.43 ± 0.00 & 0.37 ± 0.00 & 0.51 ± 0.00 & 0.24 ± 0.00 & 0.72 ± 0.00 & 0.25 ± 0.00 & 0.71 ± 0.00 \\
xgboost & 0.42 ± 0.01 & 0.32 ± 0.01 & 0.60 ± 0.01 & 0.24 ± 0.00 & 0.70 ± 0.00 & 0.25 ± 0.00 & 0.71 ± 0.01 \\
\hline
\end{tabular}


In [37]:
df[df['Model'] == 'lg'].Loss_function

14     l2
111    l2
Name: Loss_function, dtype: object

In [38]:
np.log(4)

np.float64(1.3862943611198906)

In [39]:
print(to_latex(df, metrics=[['mean_f1_elt_sup_3.0_test', 'mean_prec_elt_sup_3.0_test', 'mean_rec_elt_sup_3.0_test', 'mean_iou_elt_sup_3.0_test', 'mean_auoc_elt_sup_3.0_test'], \
                            ['mean_iou_elt_sup_3.0_test', 'mean_auoc_elt_sup_3.0_test']], loss='weightedcrossentropy',\
               targets=['nbsinister-kmeans-5-Class-Dept', 'burnedarea-kmeans-5-Class-Dept']))

\begin{tabular}{lccccccc}
\hline
Model & nbsinister-kmeans-5-Class-Dept & nbsinister-kmeans-5-Class-Dept & nbsinister-kmeans-5-Class-Dept & nbsinister-kmeans-5-Class-Dept & nbsinister-kmeans-5-Class-Dept & burnedarea-kmeans-5-Class-Dept & burnedarea-kmeans-5-Class-Dept \\
 & mean_f1_elt_sup_3.0_test & mean_prec_elt_sup_3.0_test & mean_rec_elt_sup_3.0_test & mean_iou_elt_sup_3.0_test & mean_auoc_elt_sup_3.0_test & mean_iou_elt_sup_3.0_test & mean_auoc_elt_sup_3.0_test \\
\hline
graphCastGRU & NaN & NaN & NaN & NaN & NaN & 0.10 ± 0.02 & 1.00 ± 0.00 \\
GRU & 0.91 ± 0.01 & 0.95 ± 0.05 & 0.88 ± 0.04 & 0.35 ± 0.02 & 1.00 ± 0.01 & NaN & NaN \\
LSTM & 0.94 ± 0.02 & 0.99 ± 0.01 & 0.89 ± 0.04 & 0.34 ± 0.03 & 0.99 ± 0.01 & NaN & NaN \\
NetMLP & 0.85 ± 0.03 & 0.93 ± 0.03 & 0.79 ± 0.05 & 0.31 ± 0.01 & 0.99 ± 0.01 & NaN & NaN \\
DilatedCNN & 0.90 ± 0.02 & 0.96 ± 0.04 & 0.85 ± 0.02 & 0.34 ± 0.01 & 1.00 ± 0.01 & NaN & NaN \\
\hline
\end{tabular}


In [40]:
print(to_latex(df, metrics=[['mean_normalized_f1_test', 'mean_normalized_prec_test', 'mean_normalized_rec_test', 'mean_normalized_iou_test'], \
                            ['mean_normalized_iou_test']], loss='weightedcrossentropy',\
               targets=['nbsinister-kmeans-5-Class-Dept', 'burnedarea-kmeans-5-Class-Dept']))

\begin{tabular}{lccccc}
\hline
Model & nbsinister-kmeans-5-Class-Dept & nbsinister-kmeans-5-Class-Dept & nbsinister-kmeans-5-Class-Dept & nbsinister-kmeans-5-Class-Dept & burnedarea-kmeans-5-Class-Dept \\
 & mean_normalized_f1_test & mean_normalized_prec_test & mean_normalized_rec_test & mean_normalized_iou_test & mean_normalized_iou_test \\
\hline
graphCastGRU & NaN & NaN & NaN & NaN & 0.08 ± 0.01 \\
GRU & 0.16 ± 0.02 & 0.24 ± 0.04 & 0.13 ± 0.02 & 0.09 ± 0.01 & NaN \\
LSTM & 0.16 ± 0.01 & 0.27 ± 0.01 & 0.13 ± 0.01 & 0.09 ± 0.01 & NaN \\
NetMLP & 0.15 ± 0.01 & 0.22 ± 0.02 & 0.15 ± 0.01 & 0.08 ± 0.01 & NaN \\
DilatedCNN & 0.16 ± 0.01 & 0.31 ± 0.03 & 0.12 ± 0.01 & 0.09 ± 0.01 & NaN \\
\hline
\end{tabular}


In [41]:
df[(df['Task_type'] == 'binary') & (df['Loss_function'] == 'softmax')]['rec_area_class_hard']

153    0.179
Name: rec_area_class_hard, dtype: float64

In [42]:
[[0]] + [i for i in range(4)]

[[0], 0, 1, 2, 3]